# 少榆0616ver_4 進度報告 Notebook

> 報告主軸：從老師上次看到的 `0609ver_3`，推進到 `0616ver_4`。  
> 這版不再用 README 或其他 markdown 檔當報告流程，而是直接用這份 Notebook 依序開程式碼與設定檔說明。

## 一句話總結

我這次把少榆端從「靜態 template / schema」推進到：

```text
sensor_1min / sensor_3min
-> MonitoringWorker 定時查 DB
-> threshold 判斷 warning / fault
-> mapping 對應 cause_id / response_id
-> alert_event 寫回 Database/versionB
-> alert_cause_link / alert_response_link 建立關聯
-> batch_station_status 更新
-> FutureService 寫回 future_prediction_result
-> Manager / Engineer UI 後續可透過 DB function 查詢
```

## 報告時提醒

目前這版已完成程式架構與測試腳本，但 **PostgreSQL 端到端實測需要連到 DB 環境後才可以宣稱完成**。


In [ ]:
from pathlib import Path
import json
import os
import sys

# Notebook 建議放在 少榆0616ver_4 根目錄底下。
# 若你把 Notebook 放在其他地方，可手動修改 PROJECT_DIR。
PROJECT_DIR = Path.cwd()

# 若目前工作目錄不是 少榆0616ver_4，嘗試自動尋找。
if not (PROJECT_DIR / "webservices").exists():
    candidates = list(Path.cwd().glob("**/少榆0616ver_4"))
    if candidates:
        PROJECT_DIR = candidates[0]

print("目前專案資料夾：", PROJECT_DIR)
print("webservices 是否存在：", (PROJECT_DIR / "webservices").exists())
print("rules 是否存在：", (PROJECT_DIR / "rules").exists())

def show_file(path, start=1, end=None, keyword=None):
    # 顯示指定檔案內容片段；start/end 為 1-based 行號。
    p = PROJECT_DIR / path
    print("=" * 100)
    print(path)
    print("=" * 100)
    if not p.exists():
        print("找不到檔案：", p)
        return
    text = p.read_text(encoding="utf-8", errors="replace").splitlines()
    if keyword:
        matched = [i for i, line in enumerate(text, start=1) if keyword in line]
        if matched:
            start = max(1, matched[0] - 8)
            end = min(len(text), matched[0] + 25)
        else:
            print(f"找不到 keyword: {keyword}")
            return
    if end is None:
        end = min(len(text), start + 80)
    for i, line in enumerate(text[start-1:end], start=start):
        print(f"{i:04d}: {line}")

def load_json(path, max_chars=4000):
    p = PROJECT_DIR / path
    data = json.loads(p.read_text(encoding="utf-8"))
    text = json.dumps(data, ensure_ascii=False, indent=2)
    print(text[:max_chars])
    if len(text) > max_chars:
        print("\n...（後面省略，實際報告可打開完整檔案）")


## 1. 版本定位與老師上次要求

### 這一步要講什麼

老師上次看到的版本大約是 `0609ver_3`。  
當時老師主要指出：不能只有 template、schema、JSON，必須要有一支 service 去定時檢查資料、產生 alert event，並回寫 database。

### 這一步對應的實際檔案

這一步不開 README，不開 markdown。直接用接下來的程式碼檔案說明：

| 功能 | 檔案 |
|---|---|
| 連接 Database/versionB | `webservices/integration_adapter/database_versionb_adapter.py` |
| Timer / Worker | `webservices/monitoring_worker/monitoring_worker.py` |
| threshold 判斷 | `rules/sensor_thresholds.json`、`webservices/monitoring_worker/threshold_evaluator.py` |
| sensor -> cause / response | `rules/sensor_event_mapping.json`、`webservices/monitoring_worker/detection_mapping.py` |
| alert event 回寫 | `webservices/monitoring_worker/alert_event_writer.py` |
| batch_station_status 更新 | `webservices/monitoring_worker/batch_station_status_writer.py` |
| future_prediction_result 回寫 | `webservices/future_service/future_service.py` |
| DB 實測 | `scripts/check_db_connection.py`、`scripts/run_db_smoke_test.py` |

### 可以這樣講

老師，我這次報告不從 README 開始，而是直接用 Notebook 展示程式流程。  
我這版主要解決上次您問的：**Alert event 是誰產生、何時產生、寫去哪裡、UI 後續怎麼查。**


In [ ]:
key_files = [
    "webservices/integration_adapter/database_versionb_adapter.py",
    "webservices/monitoring_worker/monitoring_worker.py",
    "webservices/monitoring_worker/threshold_evaluator.py",
    "webservices/monitoring_worker/detection_mapping.py",
    "webservices/monitoring_worker/alert_event_writer.py",
    "webservices/monitoring_worker/batch_station_status_writer.py",
    "webservices/future_service/future_service.py",
    "webservices/monitoring_worker/duplicate_alert_guard.py",
    "scripts/check_db_connection.py",
    "scripts/run_db_smoke_test.py",
    "rules/sensor_thresholds.json",
    "rules/sensor_event_mapping.json",
]
for f in key_files:
    print(f"{'OK' if (PROJECT_DIR / f).exists() else 'MISSING'}  {f}")


## 2. Integration Adapter：少榆端如何直接 import Database/versionB


### 要開啟的檔案

`webservices/integration_adapter/database_versionb_adapter.py`

### 檔案主旨

這是少榆端和 `Database/versionB` 的整合入口。

### 它做什麼

它集中處理 `Database/versionB` 的 import path，讓少榆端可以直接呼叫余宇承那邊的 DB function。  
這版不走 HTTP endpoint，不開 FastAPI，也不在少榆端自己重寫 SQL。

### 報告講法

這個檔案負責找到正式的 `Project-SprayLine-main/Database/versionB`。  
少榆端的 MonitoringWorker、FutureService、Alert writer 都會透過它呼叫 DB function。


In [ ]:
show_file("webservices/integration_adapter/database_versionb_adapter.py", start=1, end=95)


In [ ]:
# 可執行檢查：確認 adapter 狀態。
# 若在老師報告環境沒有正式 Project root，可能會 fallback 到 external/Database/versionB。
try:
    sys.path.insert(0, str(PROJECT_DIR))
    from webservices.integration_adapter.database_versionb_adapter import get_adapter_status
    print(json.dumps(get_adapter_status(), ensure_ascii=False, indent=2))
except Exception as e:
    print("adapter demo 無法執行，原因：", repr(e))
    print("報告時可只展示上方程式碼邏輯。")


## 3. MonitoringWorker：老師說的 Timer / 背景檢查程式


### 要開啟的檔案

`webservices/monitoring_worker/monitoring_worker.py`

### 檔案主旨

這就是老師上次說的 Timer / Thread / 背景檢查程式。

### 它做什麼

1. 從 `sensor_1min` / `sensor_3min` 查資料。
2. 將 sensor row 轉成 sensor payload。
3. 呼叫 threshold evaluator 判斷 warning / fault。
4. 若有異常，寫入 alert event。
5. 同步更新 batch station status。
6. 提供 `run_monitoring_once()` 與 `run_forever()` 兩種入口。

### 報告講法

老師，上次您問「資料進 DB 後，誰產生 alert event？」  
這次我補的就是 `MonitoringWorker`。它定期查 DB，偵測異常後把 alert 寫回 DB。


In [ ]:
show_file("webservices/monitoring_worker/monitoring_worker.py", start=1, end=170)


## 4. 1Hz 改成 sensor_1min / sensor_3min


### 要開啟的檔案

`webservices/monitoring_worker/monitoring_worker.py`

### 相關檔案

- `schema/sensor_1min.schema.json`
- `schema/sensor_3min.schema.json`
- `csv_templates/sensor_1min_template.csv`
- `csv_templates/sensor_3min_template.csv`

### 檔案主旨

老師上次指出舊版還有 1Hz 的設計。  
這版正式改成對齊 GitHub DB 的 `sensor_1min` 與 `sensor_3min`。

### 報告講法

MonitoringWorker 不再查舊的 1Hz，而是呼叫 Database/versionB 的 `query_sensor_1min` 和 `query_sensor_3min`。


In [ ]:
show_file("webservices/monitoring_worker/monitoring_worker.py", keyword="query_sensor_1min")


## 5. Threshold：如何判斷 warning / fault


### 要開啟的檔案

1. `rules/sensor_thresholds.json`
2. `webservices/monitoring_worker/threshold_evaluator.py`

### 檔案主旨

`sensor_thresholds.json` 定義每個 sensor 的 warning / fault 門檻。  
`threshold_evaluator.py` 實際拿 sensor 數值去判斷狀態。

### 它做什麼

目前 threshold 先保留在 JSON config，不直接修改 Database/versionB schema。  
如果之後確認 threshold 要進 DB，可以再把這層抽換成查 DB。

### 報告講法

老師，目前 threshold 我先用 JSON config 保存，避免在 DB schema 未完全確認前擅自改表。  
但 service 流程已經可以使用這些門檻去判斷 warning / fault。


In [ ]:
load_json("rules/sensor_thresholds.json")


In [ ]:
show_file("webservices/monitoring_worker/threshold_evaluator.py", start=1, end=130)


## 6. Mapping：sensor 異常如何對應原因與對策


### 要開啟的檔案

1. `rules/sensor_event_mapping.json`
2. `webservices/monitoring_worker/detection_mapping.py`

### 檔案主旨

這一層把「sensor 超標」轉成「具體故障狀態、原因、對策」。

### 它做什麼

例如：

```text
filter_diff_pressure_bar fault
-> issue_state = FILTER_CLOG
-> cause_id = FILTER_CLOG
-> response_ids = REPLACE_FILTER / BACKWASH_FILTER
-> state_field = filter_state
-> response_field = filter_response_id
```

### 報告講法

老師，上次您問原因和對策要怎麼對。  
我這次把數值門檻和原因對策分開，threshold 只負責判斷異常，mapping 負責對應 cause_id / response_id。


In [ ]:
load_json("rules/sensor_event_mapping.json")


In [ ]:
show_file("webservices/monitoring_worker/detection_mapping.py", start=1, end=220)


## 7. Alert Event Writer：alert_event 回寫 DB


### 要開啟的檔案

`webservices/monitoring_worker/alert_event_writer.py`

### 檔案主旨

這個檔案負責把偵測到的異常寫回 Database/versionB。

### 它做什麼

它不自己寫 SQL，而是呼叫：

```text
db_alert.insert_alert_event()
db_alert.link_alert_cause()
db_alert.link_alert_response()
```

寫回的 DB 目標是：

```text
alert_event
alert_cause_link
alert_response_link
```

### 報告講法

老師，這裡就是您上次說的「alert event 要回寫 database」。  
這版已經不是空 template，而是透過 Database/versionB 的 function 寫回。


In [ ]:
show_file("webservices/monitoring_worker/alert_event_writer.py", start=1, end=170)


## 8. Batch Station Status：更新 station / component 狀態


### 要開啟的檔案

`webservices/monitoring_worker/batch_station_status_writer.py`

### 檔案主旨

這個檔案把 alert 結果同步到 `batch_station_status`，讓 Manager UI 後續能知道哪個 station / component 出問題。

### 它做什麼

它呼叫：

```text
db_status.upsert_batch_station_status()
```

並更新：

```text
robot_arm_state / nozzle_state / filter_state / compressor_state / spray_width_state / quality_state
robot_arm_response_id / nozzle_response_id / filter_response_id / compressor_response_id / spray_width_response_id / quality_response_id
```

### 報告講法

老師，我不是只寫 alert_event，也同步更新 batch_station_status。  
這樣 UI 之後可以查目前 batch / station 的整體狀態。


In [ ]:
show_file("webservices/monitoring_worker/batch_station_status_writer.py", start=1, end=130)


## 9. FutureService：future_prediction_result 回寫 DB


### 要開啟的檔案

`webservices/future_service/future_service.py`

### 檔案主旨

這是少榆端 Future 預測服務。

### 它做什麼

1. 建立 future prediction payload。
2. 欄位對齊 `future_prediction_result`。
3. 透過 `db_future.insert_future_prediction_result()` 寫回 DB。

### DB 欄位

```text
prediction_id
batch_id
station_id
prediction_time
predicted_ok_rate
predicted_ng_count
quality_score
risk_level
model_input_source
created_at
```

### 報告講法

老師，上次提到預測資訊要能回到資料庫，讓 UI 使用。  
這版 FutureService 已經補 `save_future_prediction_result()`，能寫回 `future_prediction_result`。


In [ ]:
show_file("webservices/future_service/future_service.py", start=1, end=170)


In [ ]:
# 可執行 demo：不連 DB，只產生 future payload。
try:
    sys.path.insert(0, str(PROJECT_DIR))
    from webservices.future_service.future_service import build_future_prediction_payload
    payload = build_future_prediction_payload(
        batch_id="B_DEMO_NOTEBOOK",
        station_id="Station_1",
        predicted_ok_rate=88.5,
        predicted_ng_count=18,
        quality_score=0.82,
        model_input_source="notebook_demo"
    )
    print(json.dumps(payload, ensure_ascii=False, indent=2))
except Exception as e:
    print("FutureService demo 無法執行，原因：", repr(e))


## 10. Duplicate Alert Suppression：避免同一異常重複寫入


### 要開啟的檔案

1. `webservices/monitoring_worker/duplicate_alert_guard.py`
2. `webservices/monitoring_worker/config.py`

### 檔案主旨

Timer 每分鐘執行時，可能會在 lookback window 中重複看到同一筆異常。  
這個機制用來避免同一個 alert 一直重複寫入。

### 它做什麼

同一個：

```text
batch_id + station_id + sensor_name + state + cause_id
```

如果在設定時間內已有未確認 alert，就不重複寫入。

預設：

```text
SPRAYLINE_DUPLICATE_ALERT_SUPPRESSION_MINUTES=5
```

### 報告講法

老師，因為 Timer 會定期查資料，所以我補了一個防重複機制，避免 alert_event 一直暴增。


In [ ]:
show_file("webservices/monitoring_worker/duplicate_alert_guard.py", start=1, end=170)


In [ ]:
show_file("webservices/monitoring_worker/config.py", start=1, end=120)


## 11. DB 實測準備：連線檢查與 smoke test


### 要開啟的檔案

1. `scripts/check_db_connection.py`
2. `scripts/run_db_smoke_test.py`

### 檔案主旨

這兩個檔案是 PostgreSQL 實測用的。

### 它們做什麼

`check_db_connection.py`：只讀測試，不寫資料。  
用來確認 DB 連線、環境變數、Database/versionB function 是否能 import。

`run_db_smoke_test.py`：寫入測試資料，跑完整流程。  
用來確認 alert_event、alert_cause_link、alert_response_link、batch_station_status、future_prediction_result 是否能寫回 DB。

### 報告講法

目前還不能宣稱 PostgreSQL 端到端實測完成，因為要等 DB 環境可用。  
但我已經準備好實測腳本，等能連到余宇承那邊的 DB，就可以直接跑。


In [ ]:
show_file("scripts/check_db_connection.py", start=1, end=160)


In [ ]:
show_file("scripts/run_db_smoke_test.py", start=1, end=220)


## 12. 實測時要設定的環境變數

這一步不需要開 README，直接在 Notebook 說明要設定哪些環境變數。

```powershell
$env:SPRAYLINE_PROJECT_ROOT="C:\path\to\Project-SprayLine-main"

$env:DB_HOST="localhost"
$env:DB_PORT="5432"
$env:DB_USER="postgres"
$env:DB_PASSWORD="你的PostgreSQL密碼"
$env:DB_NAME="sprayline"

$env:SPRAYLINE_MONITOR_STATIONS="Station_1"
$env:SPRAYLINE_MONITOR_LOOKBACK_MINUTES="10"
$env:SPRAYLINE_DUPLICATE_ALERT_SUPPRESSION_MINUTES="5"
```

實測指令：

```powershell
python scripts\check_db_connection.py
python scripts\run_db_smoke_test.py --write-test-data --station Station_1
```


In [ ]:
# 這格只顯示目前環境變數狀態，不顯示完整密碼。
keys = [
    "SPRAYLINE_PROJECT_ROOT",
    "DB_HOST",
    "DB_PORT",
    "DB_USER",
    "DB_PASSWORD",
    "DB_NAME",
    "SPRAYLINE_MONITOR_STATIONS",
    "SPRAYLINE_MONITOR_LOOKBACK_MINUTES",
    "SPRAYLINE_DUPLICATE_ALERT_SUPPRESSION_MINUTES",
]
for k in keys:
    v = os.getenv(k)
    if k == "DB_PASSWORD" and v:
        v = "*" * min(len(v), 8)
    print(f"{k} = {v}")


## 13. 實測成功後要查哪些 SQL

若 smoke test 成功，應查以下資料表：

```sql
SELECT event_id, batch_id, station_id, sensor_name, measured_value, state, cause, ts, message
FROM alert_event
WHERE batch_id LIKE 'B_SHAOYU_E2E_%'
ORDER BY ts DESC
LIMIT 5;
```

```sql
SELECT acl.alert_id, acl.cause_id, acl.is_primary
FROM alert_cause_link acl
JOIN alert_event ae ON ae.event_id = acl.alert_id
WHERE ae.batch_id LIKE 'B_SHAOYU_E2E_%'
ORDER BY ae.ts DESC
LIMIT 5;
```

```sql
SELECT arl.alert_id, arl.response_id, arl.executed_at, arl.operator_id
FROM alert_response_link arl
JOIN alert_event ae ON ae.event_id = arl.alert_id
WHERE ae.batch_id LIKE 'B_SHAOYU_E2E_%'
ORDER BY ae.ts DESC
LIMIT 5;
```

```sql
SELECT batch_id, station_id,
       robot_arm_state, nozzle_state, filter_state,
       compressor_state, spray_width_state, quality_state,
       filter_response_id, write_time
FROM batch_station_status
WHERE batch_id LIKE 'B_SHAOYU_E2E_%'
ORDER BY write_time DESC
LIMIT 5;
```

```sql
SELECT prediction_id, batch_id, station_id,
       predicted_ok_rate, predicted_ng_count,
       quality_score, risk_level, model_input_source, created_at
FROM future_prediction_result
WHERE batch_id LIKE 'B_SHAOYU_E2E_%'
ORDER BY created_at DESC
LIMIT 5;
```

報告講法：  
老師，等我連到 DB 後，不只會看程式有沒有執行成功，也會用 SQL 查這幾張表，確認真的有寫回。


## 14. Manager UI / Engineer UI：我不改 UI，但整理資料怎麼查

### 這一步不開 markdown 檔

直接說明職責切分。

### 我的負責範圍

我負責：

```text
alert_event 回寫
alert_cause_link / alert_response_link 回寫
batch_station_status 更新
future_prediction_result 回寫
整理 UI 可查 function
```

我不負責：

```text
Manager UI 前端畫面
Engineer UI 前端畫面
WebServices/time_series_service_B 的 HTTP endpoint
```

### UI 組後續可查的 DB function

Manager UI：

```python
get_manager_summary()
get_future_prediction_summary()
get_batch_detail()
get_latest_batches()
```

Engineer UI：

```python
get_alert_detail()
get_alert_ui_card()
get_alert_causes()
get_alert_responses()
get_responses_for_cause()
get_batch_station_status()
```

報告講法：  
老師，我這邊沒有直接改 UI，因為 Manager UI / Engineer UI 是其他同學負責。  
我這邊把資料寫回 DB，並整理 UI 後續可以從 Database/versionB 查哪些 function，避免 UI hard code。


## 15. Batch Summary 目前定位

### 這一步不開 markdown 檔

直接說明目前不擅自新增 DB table。

### 目前做法

我目前不新增 `batch_summary` table，也不新增 `BatchSummaryService`。  
因為目前 Database/versionB 已有：

```text
batch_station_status
future_prediction_result
alert_event
get_manager_summary()
get_batch_detail()
get_future_prediction_summary()
```

所以目前定位是：

```text
少榆端寫入 alert_event、batch_station_status、future_prediction_result。
Manager summary 由 Database/versionB 的 query function 查詢產生。
```

報告講法：  
老師，您上次有提到 batch summary。  
我這邊先沒有擅自新增 table，因為這會影響 DB schema。  
目前我先讓少榆端寫入 alert_event、batch_station_status 和 future_prediction_result，再由 Database/versionB 的 summary function 查詢。  
如果您希望我主動產生 batch_summary event，我下一步會先跟余宇承確認 schema，再新增 BatchSummaryService。


## 16. 目前完成與尚未完成

### 已完成

```text
1. sensor 1Hz 改成 sensor_1min / sensor_3min。
2. 新增 MonitoringWorker / Timer 查 DB。
3. threshold 判斷 warning / fault。
4. sensor_event_mapping 對應 cause_id / response_id。
5. alert_event 寫回 Database/versionB。
6. alert_cause_link / alert_response_link 建立原因與對策關聯。
7. batch_station_status 更新 station component 狀態。
8. FutureService 寫回 future_prediction_result。
9. duplicate alert suppression 避免重複 alert。
10. DB connection check 和 smoke test script 已準備。
11. Manager / Engineer UI 對接 function 已整理。
```

### 尚未宣稱完成

```text
1. PostgreSQL 端到端實測尚未完成。
2. Manager UI / Engineer UI 尚未由我這邊串接。
3. data_quality_flag 最終值域尚未定案。
4. sensor_threshold table 是否正式使用尚未定案。
5. batch_summary event 是否要主動寫回 DB 尚未定案。
```

### 下一步

```text
1. 連到余宇承 / server 的 PostgreSQL。
2. 執行 check_db_connection.py。
3. 執行 run_db_smoke_test.py。
4. 用 SQL 查 alert_event、alert_cause_link、alert_response_link、batch_station_status、future_prediction_result。
5. 確認 duplicate alert suppression 是否生效。
```


## 17. 最後 1 分鐘總結稿

老師，我這次的 `少榆0616ver_4` 是從上次 `0609ver_3` 的 template / schema 往 service 實作推進。

我主要完成：

```text
1. 把原本的 1Hz 改成 Database/versionB 的 sensor_1min / sensor_3min。
2. 補 MonitoringWorker，定時查 DB。
3. 用 threshold 判斷 warning / fault。
4. 產生 alert_event 並寫回 Database/versionB。
5. 建立 alert_cause_link / alert_response_link，讓 alert 可以對到原因和對策。
6. 更新 batch_station_status。
7. FutureService 可以寫回 future_prediction_result。
8. 補 duplicate alert suppression，避免同一異常重複寫入。
9. 補 DB 連線測試和 smoke test，等 DB 環境可用後就能做端到端實測。
10. 補 Manager / Engineer UI 對接 function 表，讓 UI 組知道要查哪些 DB function。
```

目前還差 PostgreSQL 實際端到端測試，這部分我已經把腳本和流程準備好了。
